# Single-head causal attention, by hand

This is my first pass at self-attention, written before any polished version exists in the repo. I want to see every step and every shape, so I print the shape after each operation. Nothing here is optimised. The aim is to understand it properly.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)  # same numbers every run, so I can compare outputs
# small on purpose so I can actually read the tensors when I print them
B = 2   # batch
T = 4   # sequence length
C = 8   # embedding size
head_size = 4  # one head, so this is just how wide Q, K, V are

print(B, T, C, head_size)

2 4 8 4


## The input

`x` stands in for a batch of embedded tokens, with shape (batch, time, channels). I use random numbers for now, because the point is to follow the shapes, not to learn anything yet.

In [2]:
x = torch.randn(B, T, C)
print(x.shape) # (B, T, C) = (2, 4, 8)

torch.Size([2, 4, 8])


## Queries, keys and values

I pass the same `x` through three separate linear layers. The query is what a token is looking for, the key is what a token offers, and the value is what it hands over if it is chosen. I leave out the bias so the layers stay simple, and I project down to `head_size` so each head works in a smaller space.

In [3]:
query = nn.Linear(C, head_size, bias=False)
key   = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

q = query(x)
k = key(x)
v = value(x)

print(q.shape)  # (2, 4, 4)
print(k.shape)
print(v.shape)

torch.Size([2, 4, 4])
torch.Size([2, 4, 4])
torch.Size([2, 4, 4])


## Scores

Every query is compared with every key using a dot product, so a high score means a strong match. That gives one score for each pair of tokens, which is why the result is T by T for each item in the batch.

In [4]:
scores = q @ k.transpose(-2, -1)
print(scores.shape)  # (2, 4, 4)  one score per token pair

torch.Size([2, 4, 4])


## Scaling

I divide the scores by the square root of `head_size`. A dot product grows with the number of dimensions, and large scores push the softmax towards a single hard winner, which makes learning harder. Scaling keeps the scores in a sensible range.

In [5]:
print(scores.var())
scores = scores / head_size**0.5
print(scores.var())  # should drop by about head_size

tensor(0.5081, grad_fn=<VarBackward0>)
tensor(0.1270, grad_fn=<VarBackward0>)


## The Causal Mask

A token must not look at the future, otherwise the model could simply copy the next word while training and learn nothing. I build a lower-triangular matrix and set every position above the diagonal to minus infinity. After the softmax, those positions become exactly zero.

In [6]:
tril = torch.tril(torch.ones(T, T))
scores = scores.masked_fill(tril == 0, float("-inf"))

print(tril)
print(scores[0, 0])  # first token: only itself is finite

tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])
tensor([-0.0632,    -inf,    -inf,    -inf], grad_fn=<SelectBackward0>)


## Softmax

Softmax along the last dimension turns each row of scores into weights that add up to one. Row `i` now says how much token `i` draws from each token up to and including itself.

In [7]:
weights = F.softmax(scores, dim=-1)

print(weights.sum(dim=-1))          # all 1s
print(weights[0].triu(diagonal=1))  # zeros above the diagonal

tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SumBackward1>)
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]], grad_fn=<TriuBackward0>)


## Weighted sum

Last, I use the weights to take a weighted average of the values. Each token's output is a blend of the values from the tokens it is allowed to see, so it carries information from its context. The output shape matches the value shape, (batch, time, head_size).

In [8]:
out = weights @ v
print(out.shape)  # (2, 4, 4)  same as v

torch.Size([2, 4, 4])


## Proving the mask works

I do not want to trust the mask by eye, so I test it. If I change the last token of the input, every earlier output must stay exactly the same, because the past cannot see the future. If any earlier row changes, there is a leak.

In [9]:
def causal_attn(x):
    q = query(x)
    k = key(x)
    v = value(x)
    scores = q @ k.transpose(-2, -1) / head_size**0.5
    tril = torch.tril(torch.ones(x.size(1), x.size(1)))
    scores = scores.masked_fill(tril == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ v

out1 = causal_attn(x)

x2 = x.clone()
x2[:, -1, :] = torch.randn(B, C)
out2 = causal_attn(x2)

assert torch.allclose(out1[:, :-1, :], out2[:, :-1, :])
print("past outputs unchanged")

past outputs unchanged


## What this leaves out

This is one head with no learned position information. Multi-head attention, positional encoding (including RoPE) and dropout come next, once I have properly read up on them. I will tidy this into a proper module in the model file, and keep this notebook as it is.